In [ ]:
import sys

sys.path.append("C:/cloud_backend/langchain_app")

from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, SystemMessage
from memory_history import get_session_history, reset_session_history
from dotenv import load_dotenv

load_dotenv("../../.env")

model = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
# 메시지 기록 요약용 프롬프트와 함수 선언

summarization_prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="chat_history"),
    (
        "user",
        "위 대화 내용을 핵심 정보 위주로 간결하게 요약해 주세요. "
        "기존 요약본이 있다면 그 내용도 누락 없이 통합해서 하나의 요약문으로 작성해 주세요.",
    ),
])
summarization_chain = summarization_prompt | model


def summarize_messages(chain_input) -> list[BaseMessage]:
    # chain_input => {"input": 사용자 입력 메세지, "session_id": 세션ID} 
    """대화 기록이 일정 개수를 초과하면 요약본 + 최근 대화만 남기고 스토어를 갱신합니다."""
    session_id = chain_input["session_id"]
    stored_history = get_session_history(session_id)
    stored_messages = stored_history.messages

    # 메시지가 6개(3턴) 이하이면 요약하지 않고 통과
    if len(stored_messages) <= 6:
        return stored_messages

    # 최근 2개 메시지(1턴)는 맥락 유지를 위해 남기고, 이전 메시지들만 요약
    messages_to_summarize = stored_messages[:-2]
    recent_messages = stored_messages[-2:]

    summary_message = summarization_chain.invoke({
        "chat_history": messages_to_summarize
    })

    # 요약 내용을 SystemMessage로 변환하여 최근 메시지와 함께 스토어에 덮어쓰기
    new_messages = [
        SystemMessage(content=f"이전 대화 요약: {summary_message.content}")
    ] + recent_messages

    reset_session_history(session_id, new_messages) #type:ignore
    return new_messages


In [3]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 모든 질문에 최선을 다해 답변하십시오."),
        ("placeholder", "{chat_history}"),  # 이전 대화 이력
        ("human", "{input}"),  # 사용자의 새로운 질문
    ]
)

# 대화 실행 전 요약 함수를 먼저 거치도록 파이프라인 연결
chain_with_summarization = (
    RunnablePassthrough.assign(chat_history=summarize_messages) | prompt | model
)

# RunnableWithMessageHistory 클래스를 사용해 체인을 감쌉니다
chain_with_message_history = RunnableWithMessageHistory(
    chain_with_summarization,
    get_session_history,
    input_messages_key="input",  # 입력 메시지의 키 설정
    history_messages_key="chat_history",  # 대화 이력의 키 설정
)

c:\cloud_backend\langchain_app\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [4]:
print(chain_with_message_history.invoke(
    {"input": "안녕? 내 이름은 김철수야", "session_id": "user01"},  # 사용자의 질문
    {"configurable": {"session_id": "user01"}}  # 세션 ID 설정
).content)

안녕하세요, 김철수님! 만나서 반갑습니다. 금융 관련하여 궁금한 점이 있으면 언제든지 질문해 주세요. 어떻게 도와드릴 수 있을까요?


In [5]:
print(chain_with_message_history.invoke(
    {"input": "내 이름이 뭐라고 했지?", "session_id": "user01"},  # 사용자의 질문
    {"configurable": {"session_id": "user01"}}  # 세션 ID 설정
).content)

당신의 이름은 김철수입니다. 계속해서 도움이 필요하시면 말씀해 주세요!


In [6]:
get_session_history("user01").messages

[HumanMessage(content='안녕? 내 이름은 김철수야', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 김철수님! 만나서 반갑습니다. 금융 관련하여 궁금한 점이 있으면 언제든지 질문해 주세요. 어떻게 도와드릴 수 있을까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 40, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_74833a8d49', 'id': 'chatcmpl-EmA4bEjMyTKLROU76eUtr2AlARQIZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None, 'normalized_finish_reason': 'stop'}, id='lc_run--01a1f71c-0992-7762-abc8-767a17d8b2c0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens'

In [7]:
print(chain_with_message_history.invoke(
    {"input": "주식과 채권을 비교해줘", "session_id": "user01"},  # 사용자의 질문
    {"configurable": {"session_id": "user01"}}  # 세션 ID 설정
).content)

주식과 채권은 투자 수단으로 각각 특성과 장단점이 있습니다. 아래는 두 가지를 비교한 내용입니다:

### 1. **정의**
- **주식**: 기업의 소유권을 나타내는 증권으로, 주식을 보유한 사람은 해당 기업의 주주가 됩니다. 주주는 기업의 이익에 참여할 수 있는 권리가 있으며, 의결권도 가질 수 있습니다.
- **채권**: 정부, 기업 등이 자금을 조달하기 위해 발행한 차용증서로, 채권 보유자는 발행자에게 정해진 이자(쿠폰)를 받을 권리가 있으며, 만기일에 원금을 반환받을 수 있습니다.

### 2. **수익원**
- **주식**: 주가 상승과 배당금으로 수익을 얻을 수 있습니다. 주가는 시장의 수요와 공급에 따라 변동성이 크기 때문에 수익이 높거나 낮을 수 있습니다.
- **채권**: 정기적으로 이자 수익을 제공하며, 만기에 원금을 돌려받습니다. 채권의 수익은 상대적으로 안정적이고 예측 가능하지만, 주식보다 일반적으로 낮습니다.

### 3. **위험**
- **주식**: 기업의 성과, 시장의 변동성 등에 따라 가치가 급격히 변할 수 있으므로 위험이 높습니다. 만약 기업이 파산하면 주식 가치는 제로가 될 수 있습니다.
- **채권**: 채권은 비교적 안정적이며 특히 신뢰할 수 있는 정부에서 발행한 경우 위험이 낮습니다. 그러나 금리 상승, 발행자의 신용 등급 하락 등으로 가치가 하락할 수 있습니다.

### 4. **유동성**
- **주식**: 거래소에서 쉽게 거래 가능하여 유동성이 높습니다. 필요할 때 즉시 매도할 수 있습니다.
- **채권**: 일반적으로 주식보다 유동성이 낮습니다. 특정 채권은 만기까지 보유하는 것이 일반적이며, 거래가 활발하지 않은 경우도 있습니다.

### 5. **투자 목적**
- **주식**: 장기적인 성장과 자본 이득을 추구하는 투자자에게 적합합니다. 기업의 성장 가능성을 믿고 투자하는 경우가 많습니다.
- **채권**: 안전한 수익을 원하거나 안정적인 현금 흐름이 필요한 투자자에게 적합합니다.

이와 같이 주식과 채권은 각

In [8]:
print(chain_with_message_history.invoke(
    {"input": "초보자 입장에서 주식투자 시 주의사항을 알려줘", "session_id": "user01"},  # 사용자의 질문
    {"configurable": {"session_id": "user01"}}  # 세션 ID 설정
).content)

주식 투자를 시작하는 초보자에게는 다음과 같은 주의사항이 중요합니다:

### 1. **기본 지식 습득**
   - **주식의 기초 이해**: 주식이 무엇인지, 기업의 재무제표(손익계산서, 대차대조표 등)의 의미, 주가의 변화 요인 등을 공부하세요.
   - **시장 동향 파악**: 주식 시장의 기본 원리와 트렌드를 이해하는 것이 중요합니다.

### 2. **대상 기업 분석**
   - **기업 조사**: 투자를 고려 중인 기업에 대한 정보를 수집하고 분석하세요. 사업 모델, 경쟁력, 재무 상태 등을 검토하는 것이 필요합니다.
   - **산업 동향**: 속한 산업의 전망과 변화에 대한 정보를 확보하세요. 산업이 성장하는 추세인지, 불황인지 알고 투자 결정을 하는 것이 중요합니다.

### 3. **분산 투자**
   - **자산 분산**: 여러 기업의 주식에 분산 투자하여 위험을 줄이세요. 특정 기업이나 산업에만 집중 투자하면 위험도가 높아질 수 있습니다.

### 4. **장기 투자 관점**
   - **단기 변동성 인내**: 주식 시장은 변동성이 크므로 단기적인 주가 변동에 일희일비하지 않도록 합니다. 장기적인 투자 목표를 세우세요.
   - **시장 타이밍 피하기**: 언제 매수 또는 매도할지를 결정하는 것이 어렵기 때문에 시장 타이밍을 노리는 것보다는 정기적으로 투자하는 것이 좋습니다.

### 5. **정서적 결정 피하기**
   - **감정 관리**: 두려움이나 탐욕에 따라 투자 결정을 내리지 않도록 하세요. 감정에 휘둘리면 비합리적인 판단을 할 수 있습니다.
   - **계획 세우기**: 매매 계획을 세우고 그에 따라 실행하세요. 손절매 기준과 목표 가격을 미리 정해놓는 것이 도움이 됩니다.

### 6. **정보의 출처 확인**
   - **신뢰할 수 있는 정보**: 다양한 매체를 통해 정보를 수집하되, 신뢰할 수 있는 출처에서 얻은 정보를 바탕으로 투자 결정을 하세요.

### 7. **투자 금액 설정**
   - **자산 배분**: 전체 

In [9]:
print(chain_with_message_history.invoke(
    {"input": "채권의 종류를 알려줘", "session_id": "user01"},  # 사용자의 질문
    {"configurable": {"session_id": "user01"}}  # 세션 ID 설정
).content)

채권은 여러 종류가 있으며, 발행 주체, 만기 기간, 이자 지급 방식 등에 따라 다양한 분류가 가능합니다. 대표적인 채권의 종류는 다음과 같습니다:

### 1. **발행 주체에 따른 분류**
   - **국채 (Government Bonds)**: 국가가 발행하는 채권으로, 상대적으로 안전한 투자로 간주됩니다. 예를 들어, 미국 재무부 채권(Treasury Bonds)이 이에 해당합니다.
   - **지방채 (Municipal Bonds)**: 지방 정부나 지방 자치단체가 발행하는 채권으로, 일반적으로 세금 면제 혜택이 있습니다.
   - **회사채 (Corporate Bonds)**: 기업이 자금을 조달하기 위해 발행하는 채권입니다. 위험도는 기업의 신용도에 따라 다르며, 일반적으로 국채보다 높은 이자 수익을 제공합니다.

### 2. **만기 기간에 따른 분류**
   - **단기 채권 (Short-term Bonds)**: 만기가 1년 이하인 채권으로, 보통 단기 자금을 필요로 하는 기업이나 정부가 발행합니다.
   - **중기 채권 (Intermediate-term Bonds)**: 만기가 1년에서 10년 사이인 채권입니다.
   - **장기 채권 (Long-term Bonds)**: 만기가 10년 이상인 채권으로, 보통 안정적인 수익을 원할 때 고려됩니다.

### 3. **이자 지급 방식에 따른 분류**
   - **이자 지급 채권 (Coupon Bonds)**: 정기적으로 이자를 지급하는 채권으로, 만기 시에 원금이 반환됩니다. 이자 지급 주기는 연간, 반기, 분기 등이 있을 수 있습니다.
   - **제로 쿠폰 채권 (Zero-Coupon Bonds)**: 이자를 지급하지 않고, 할인된 가격으로 발행하여 만기 시에 원금만 지급합니다. 이자 수익은 할인된 가격과 만기 시 가치의 차이에서 발생합니다.
  
### 4. **신용 위험에 따른 분류**
   - **Investment Grade Bonds**: 신용 등급이 높은 안전한 채권으로, 투자 

In [10]:
print(chain_with_message_history.invoke(
    {"input": "금리와 채권의 관계를 알려줘", "session_id": "user01"},  # 사용자의 질문
    {"configurable": {"session_id": "user01"}}  # 세션 ID 설정
).content)

금리와 채권의 관계는 매우 밀접합니다. 다음은 그 주요 요점입니다:

### 1. **수익률과 금리의 관계**
   - **금리가 올라가면 채권 수익률도 증가**: 새로운 채권이 높은 금리로 발행되면, 기존에 발행된 낮은 금리 채권의 매력도가 감소하게 됩니다. 이로 인해 기존의 채권 가격이 하락하게 되어, 기존 채권의 수익률(투자자가 채권 매입가 대비 받게 되는 이자 수익률)이 상승하게 됩니다.
   - **금리가 내려가면 채권 수익률도 감소**: 반대로 금리가 하락하면 새로운 채권이 낮은 금리로 발행되고, 기존 채권의 가격이 오르며 수익률이 떨어집니다.

### 2. **채권 가격과 금리의 반비례 관계**
   - **가격과 수익률의 관계**: 채권의 가격은 금리와 반비례하므로, 금리가 상승할 때 채권 가격은 하락하고, 금리가 하락할 때는 채권 가격이 상승합니다.
   - **예시**: 만약 10년 만기 1,000달러 액면가의 채권이 5% 이자를 지급한다고 가정했을 때, 금리가 3%로 내려가면 이 채권은 5%의 이자를 주므로 더 매력적이 되어 가격이 상승합니다. 반대로 금리가 7%로 상승하면 이 채권은 상대적으로 덜 매력적이 되어 가격이 하락합니다.

### 3. **금리가 경제에 미치는 영향**
   - **금리에 따른 경제 상황**: 중앙은행이 금리를 인상하면 경제에 부정적인 압력이 가해질 수 있으며, 이는 기업의 차입 비용 증가 및 소비자 신뢰 하락으로 이어질 수 있습니다. 반대로 금리를 인하하면 차입 여건이 좋아져 경제를 부양하는 효과가 있습니다.
   - **인플레이션과 금리**: 인플레이션이 상승하면 중앙은행은 금리를 인상하여 물가 상승을 억제하려 할 수 있습니다. 이는 채권 시장에 직접적인 영향을 미칩니다.

### 4. **투자자의 결정**
   - 채권 투자자는 금리가 상승할 것으로 예상되면 채권을 매도하거나 새로운 채권 투자에 신중해지며, 반대로 금리가 하락할 것으로 예상되면 채권의 매력이 증가하게 됩니다.

결론적으로, 금리와 채권은 상호

In [11]:
print(chain_with_message_history.invoke(
    {"input": "회사채는 뭐지?", "session_id": "user01"},  # 사용자의 질문
    {"configurable": {"session_id": "user01"}}  # 세션 ID 설정
).content)

회사채( corporate bond )는 기업이 자금을 조달하기 위해 발행하는 채권으로, 투자자에게 원금과 이자를 지급할 것을 약속하는 금융 상품입니다. 회사채에 대한 주요 특징은 다음과 같습니다:

### 1. **발행 목적**
   - 기업은 운영 자금, 프로젝트 투자, 부채 상환 등 다양한 목적으로 회사채를 발행하여 필요한 자금을 조달합니다.

### 2. **수익**
   - **이자 지급**: 회사채는 정해진 주기(통상적으로 연 1회 또는 2회)마다 이자를 지급하며, 이자율(쿠폰 금리)은 채권 발행 시 결정됩니다. 이자 수익은 투자자에게 정기적인 현금 흐름을 제공합니다.
   - **원금 상환**: 만기일에 투자자는 액면가(원금)를 상환받습니다.

### 3. **위험**
   - **신용 위험**: 회사채는 발행한 기업의 신용도에 따라 위험 수준이 달라집니다. 기업이 파산하거나 이자 지급이 불가능해질 경우 투자자는 손실을 입을 수 있습니다. 이를 평가하기 위해 신용 평가 기관이 기업에 신용 등급을 부여합니다.
   - **시장 위험**: 금리 변동, 경제 상황 등의 외부 요인에 의해 회사채의 시장 가격이 변동할 수 있습니다.

### 4. **유형**
   - **일반 회사채**: 특정 자산 담보 없이 발행됩니다.
   - **담보 회사채**: 특정 자산(예: 부동산, 기계 등)을 담보로 하는 채권으로, 담보가 제공되므로 신용 위험이 낮습니다.
   - **전환사채**: 일정 조건에 따라 주식으로 전환할 수 있는 권리가 부여된 채권입니다.
   - **이자율 스왑(변동 이자율)**: 변동 이자율로 이자 지급이 이루어지는 회사채입니다.

### 5. **채권 시장에서의 위치**
   - 회사채는 일반적으로 정부채보다 더 높은 수익률을 제공하지만, 위험도도 상대적으로 더 높습니다. 투자자들은 적절한 위험 수용 능력에 따라 회사채를 포트폴리오에 포함시킬 수 있습니다.

회사는 회사채를 발행하여 필요한 자금을 조달하면서, 투자자는 안정적인 이자 수익을 기대

In [ ]:
# 대화기록이 요약되므로 처음 이야기한 이름은 기록에서 사라질 수 있다.

print(chain_with_message_history.invoke(
    {"input": "내 이름이 뭐라고 했지?", "session_id": "user01"},  # 사용자의 질문
    {"configurable": {"session_id": "user01"}}  # 세션 ID 설정
).content)

죄송합니다만, 이전 대화에서 귀하의 이름을 언급하지 않으셨습니다. 귀하의 이름을 알려주시면 맞춤형 답변을 드리는 데 도움이 될 것입니다. 더 궁금한 점이 있으면 언제든지 질문해 주세요!


In [13]:
get_session_history("user01").messages

[SystemMessage(content='이전 대화 요약: ### 주식과 채권 비교 및 금리와 채권의 관계 요약\n\n#### 1. **주식 vs. 채권**\n- **정의**:\n  - **주식**: 기업 소유권을 나타내며, 주주는 기업의 이익에 참여할 권리가 있음.\n  - **채권**: 자금을 조달하기 위한 차용증서로, 정기적인 이자 수익과 만기 시 원금 상환을 약속받음.\n\n- **수익원**:\n  - **주식**: 주가 상승과 배당금으로 수익, 변동성이 큼.\n  - **채권**: 안정적인 이자 수익과 원금 반환, 일반적으로 낮은 수익률.\n\n- **위험**:\n  - **주식**: 가치 변동이 크고, 파산 시 0이 될 수 있어 위험이 높음.\n  - **채권**: 상대적으로 낮은 위험이나 금리 변화와 신용 등급 하락 시 가치 하락 가능성 존재.\n\n- **유동성**:\n  - **주식**: 거래소에서 쉽게 거래 가능, 유동성이 높음.\n  - **채권**: 일반적으로 유동성이 낮아 만기까지 보유하는 경우가 많음.\n\n- **투자 목적**:\n  - **주식**: 장기 성장과 자본 이득 추구.\n  - **채권**: 안정적인 수익과 현금 흐름 필요.\n\n#### 2. **금리와 채권의 관계**\n- **금리와 채권 수익률**: 금리가 상승하면 채권 수익률도 증가하고 기존 채권 가격은 하락하며, 금리가 하락하면 수익률은 감소하고 채권 가격은 상승함.\n- **가격과 금리의 반비례 관계**: 채권 가격은 금리와 반비례하여 움직임.\n- **경제와 금리**: 중앙은행의 금리 인상은 경제에 부정적 영향을 미칠 수 있으며, 금리가 인하되면 차입 여건이 개선됨.\n- **투자자의 결정**: 금리 상승 예측 시 채권 매도 또는 신규 투자 신중, 반대로 금리 하락 예측 시 채권 매력 증가.\n\n이 내용을 통해 주식과 채권의 차이점, 투자 시 고려해야 할 점, 그리고 금리 변화에 따른 채권의 반응을 이해할 수 있습니다. 추가 질문이 있다면 언제든지 말씀해 

In [14]:
# 요약된 대화를 기반으로 새로운 질문에 응답
print(chain_with_message_history.invoke(
    {"input": "우리가 어떤 이야기를 하고 있었지?", "session_id": "user01"},  # 사용자의 질문
    {"configurable": {"session_id": "user01"}}  # 세션 ID 설정
).content)


우리는 주식과 채권의 차이점, 투자 시 고려해야 할 점, 그리고 금리 변화에 따른 채권의 반응에 대해 이야기하고 있었습니다. 특히, 주식과 채권의 정의, 수익원, 위험, 유동성, 투자 목적 등을 비교하였고, 금리가 채권의 가격과 수익률에 미치는 영향에 대해서도 논의했습니다. 이후에 질문하신 내용은 회사채에 관한 것으로, 회사채의 정의, 수익, 위험 등의 특징에 대해 설명드렸습니다. 추가적으로 더 궁금하신 것이 있다면 언제든지 질문해 주세요!
